# Entrenamiento: pronóstico semanal directo por municipio y enfermedad (`rf-municipal-v2`)
EDA sobre `casos_municipio_semana.csv`, features municipales sin fuga, entrenamiento por patología, comparación contra persistencia/SARIMA, backtest y SHAP.
El pipeline reproducible vive en `model/train.py` (este notebook reutiliza sus funciones).

In [ ]:
import os, sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('.'))
import train as T
raw = T.load_raw(T.DATA)
raw.groupby('enfermedad').agg(casos=('casos', 'sum'), inicio=('semana', 'min'), fin=('semana', 'max'), semanas=('semana', 'nunique'))

## 1. Series históricas reales por enfermedad (Región Centro, 12 municipios)

In [ ]:
reg = raw.groupby(['enfermedad', 'fecha_inicio_semana'], as_index=False)['casos'].sum()
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=False)
for ax, e in zip(axes, T.ENFERMEDADES):
    g = reg[reg.enfermedad == e]
    ax.plot(g.fecha_inicio_semana, g.casos)
    ax.set_title(f'Casos semanales acumulados (12 municipios) - {e}')
plt.tight_layout(); plt.show()

## 2. Features municipales y partición temporal sin fuga por enfermedad

In [ ]:
mun_map = T.build_mun_map(raw)
featured = T.build_features(raw, mun_map, with_target=True)
labeled = featured.dropna(subset=T.FEATURES + ['log_next']).reset_index(drop=True)
print('Features:', len(T.FEATURES), T.FEATURES)

## 3. Resultados de entrenamiento y backtest (`metricas.json` / `backtest.json`)

In [ ]:
metricas = json.load(open(os.path.join(T.ART, 'metricas.json'), encoding='utf-8'))
pd.DataFrame({
    k: {
        'MAE RF': d['rf']['mae'],
        'MAE persist': d['persistencia']['mae'],
        'MAE SARIMA': d['sarima']['mae'],
        'Exactitud RF': round(d['rf']['exactitud'] * 100, 1),
        'F1 RF': d['rf']['f1_macro'],
    }
    for k, d in metricas['test']['municipal'].items()
}).T.round(3)

In [ ]:
bt = json.load(open(os.path.join(T.ART, 'backtest.json'), encoding='utf-8'))
pd.DataFrame({k: {'MAE RF': d['rf']['mae'], 'MAE persist': d['persistencia']['mae'], 'Exactitud %': round(d['rf']['exactitud'] * 100, 1)} for k, d in bt['agregado'].items()}).T.round(3)

## 4. Explicabilidad (SHAP)

In [ ]:
from IPython.display import Image
Image(os.path.join(T.DOCS, 'shap_summary.png'))